# Real to Comic Photo — CycleGAN modelling

This notebook walks through the dataset, the model and training. All the logic lives in the `comicgan` package; the notebook just calls it.

- **Domain A (photo)**: `faces/`, real face photos
- **Domain B (comic)**: `comics/`, comic-style versions

Training outputs:
- `models/photo2comic.keras` and `models/comic2photo.keras` (used by the Streamlit app)
- `outputs/samples/epoch_XXX.png` (preview grids), `outputs/history.csv`, `outputs/checkpoints/`

The same training can run from the terminal: `python -m comicgan.train --epochs 50`.

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

from comicgan.config import TrainConfig
from comicgan.data import list_domain_files
from comicgan.model import build_discriminator, build_generator

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## Configuration

The defaults match the original Kaggle run (128 px, batch 16, 6 residual blocks, generator LR 1e-4, discriminator LR 4e-4).
For a quick test, set something like `epochs=1, limit=64, batch_size=4`.

In [ ]:
cfg = TrainConfig(epochs=50)
cfg

## Dataset

Face2Comics v2 by Sxela: 10,000 photos and 10,000 comics, 1024×1024. The files are paired by name (`faces/0.jpg` ↔ `comics/0.jpg`), but CycleGAN does not need the pairing, so each domain is shuffled on its own during training.

In [ ]:
photos, comics = list_domain_files(cfg.data_dir)
print(f"photos: {len(photos)}, comics: {len(comics)}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(Image.open(photos[i]))
    axes[1, i].imshow(Image.open(comics[i]))
    axes[0, i].axis("off")
    axes[1, i].axis("off")
axes[0, 0].set_title("photo", loc="left")
axes[1, 0].set_title("comic", loc="left")
plt.tight_layout()

## Model

- **Generator**: 3 conv encoder blocks (down to 1/4 resolution), N residual blocks, 2 transposed-conv decoder blocks with skip connections to the encoder, then a 7×7 conv with `tanh` output. Instance norm is `GroupNormalization(groups=-1)`.
- **Discriminator**: PatchGAN that outputs a 14×14 grid of real/fake logits for a 128 px input.
- **Losses**: adversarial BCE (label smoothing 0.1 for the discriminator), cycle-consistency L1 (λ=10) and identity L1 (λ/2).

In [ ]:
generator = build_generator(cfg.transformer_blocks)
discriminator = build_discriminator()
print(f"Generator params:     {generator.count_params():,}")
print(f"Discriminator params: {discriminator.count_params():,}")

## Training

In the original run, one epoch (618 steps) took about 60 s on a Kaggle TPU v3-8. Expect it to be much slower on CPU, so use a GPU (Kaggle/Colab) for the full 50 epochs.

In [ ]:
from comicgan.train import train

history = train(cfg)

In [ ]:
plt.figure(figsize=(10, 4))
for name, values in history.history.items():
    plt.plot(values, label=name)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Training losses");

## Samples

Each row: photo | generated comic | comic | generated photo.

In [ ]:
latest = sorted((cfg.output_dir / "samples").glob("epoch_*.png"))[-1]
print(latest.name)
Image.open(latest)

## Inference

This loads the exported model the same way the Streamlit app does.

In [ ]:
from comicgan.config import PHOTO2COMIC_FILE
from comicgan.inference import load_generator, translate

photo2comic = load_generator(cfg.model_dir / PHOTO2COMIC_FILE)
source = Image.open(photos[-1])
result = translate(photo2comic, source, size=256)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, img, title in zip(axes, [source, result], ["input", "comic"]):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")